# WeatherMesh-3 inference
Raw `huge/` sample -> encoder inputs -> model -> de-normalized ERA5 forecast. The input processing (which the repo withholds) is reconstructed from the mesh definitions, reusing `meshes.LatLonGrid`, `utils.interp_levels`, and `model.get_WeatherMesh3`. Runs on Linux + CUDA (fused NATTEN attention).

## Environment

In [ ]:
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt


def find_repo():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "huge").is_dir() and (base / "constants").is_dir():
            return base
    return Path.cwd()


REPO = find_repo()
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

DATA = REPO / "huge" / "proc" / "haoxing_data" / "wm3" / "data"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("repo:", REPO)
print("torch:", torch.__version__, "| device:", device)

## Input and output meshes
Same construction as `model.get_WeatherMesh3`. Both inputs resolve to 28 (`levels_medium`) internal levels and `n_vars = 157`.

In [ ]:
from meshes import LatLonGrid
from utils import levels_gfs, levels_hres, levels_medium, interp_levels, get_date

extra_in = ["45_tcc", "168_2d", "246_100u", "247_100v"]
extra_out = ["15_msnswrf", "45_tcc", "168_2d", "246_100u", "247_100v",
             "142_lsp", "143_cp", "201_mx2t", "202_mn2t",
             "142_lsp-6h", "143_cp-6h", "201_mx2t-6h", "202_mn2t-6h"]

gfs_mesh = LatLonGrid(source="neogfs-25", extra_sfc_vars=extra_in,
                      extra_sfc_pad=len(extra_out) - len(extra_in),
                      input_levels=levels_gfs, levels=levels_medium)
hres_mesh = LatLonGrid(source="neohres-20", extra_sfc_vars=extra_in,
                       extra_sfc_pad=len(extra_out) - len(extra_in),
                       input_levels=levels_hres, levels=levels_medium)
era_mesh = LatLonGrid(source="era5-28", extra_sfc_vars=extra_out, levels=levels_medium)

for name, m in [("neogfs", gfs_mesh), ("neohres", hres_mesh), ("era5-out", era_mesh)]:
    print(f"{name:9} input_levels={len(m.input_levels):2} internal_levels={m.n_levels} "
          f"n_pr={m.n_pr} n_sfc={m.n_sfc} n_vars={m.n_vars} shape={tuple(m.shape())}")

## Load the raw sample

In [ ]:
def load_source(source):
    p = next((DATA / source / "f000").glob("**/*.npz"))
    z = np.load(p)
    extras = [np.load(next((DATA / source / "extra" / v).glob("**/*.npz")))["x"]
              for v in extra_in]
    return z["pr"], z["sfc"], np.stack(extras, axis=-1), int(p.stem)


pr_g, sfc_g, ex_g, TS = load_source("neogfs")
pr_h, sfc_h, ex_h, _ = load_source("neohres")

print("neogfs  pr", pr_g.shape, "sfc", sfc_g.shape, "extra", ex_g.shape)
print("neohres pr", pr_h.shape, "sfc", sfc_h.shape, "extra", ex_h.shape)
print("timestamp", TS, "->", get_date(TS).strftime("%Y-%m-%d %H:%MZ"))

## Reconstruct the encoder inputs
Trim the south pole (721 -> 720), lay pressure out variable-major / level-minor, append `[core sfc | extra sfc | zeropad]`, then lift the native levels (25 / 20) to the 28 internal levels with `interp_levels`.

In [ ]:
def build_input(mesh, pr, sfc, ex):
    pr = pr[:720].astype(np.float32)
    sfc = sfc[:720].astype(np.float32)
    ex = ex[:720].astype(np.float32)
    H, W, nv, nl = pr.shape
    pr_flat = pr.reshape(H, W, nv * nl)
    pad = np.zeros((H, W, mesh.extra_sfc_pad), np.float32)
    sfc_block = np.concatenate([sfc, ex, pad], axis=-1)
    x_in = torch.from_numpy(np.concatenate([pr_flat, sfc_block], axis=-1))
    return interp_levels(x_in, mesh, mesh.input_levels, mesh.levels)


gx = build_input(gfs_mesh, pr_g, sfc_g, ex_g)
hx = build_input(hres_mesh, pr_h, sfc_h, ex_h)
t0s = torch.tensor([TS])

zpad = gx[..., gfs_mesh.n_pr + len(gfs_mesh.core_sfc_vars) + len(extra_in):]
print("gfs  input tensor:", tuple(gx.shape))
print("hres input tensor:", tuple(hx.shape))
print("zeropad channels :", tuple(zpad.shape), "all zero:", bool(torch.all(zpad == 0)))

## Verify the reconstruction
De-normalize with each mesh's own `normalization_matrix_mean/std` and confirm the fields are physical.

In [ ]:
def denorm_channel(x, mesh, idx):
    return x[..., idx].numpy() * mesh.normalization_matrix_std[idx] + mesh.normalization_matrix_mean[idx]


i2t = gfs_mesh.n_pr + gfs_mesh.core_sfc_vars.index("167_2t")
iz500 = gfs_mesh.pressure_vars.index("129_z") * gfs_mesh.n_levels + gfs_mesh.levels.index(500)
t2m = denorm_channel(gx, gfs_mesh, i2t)
z500 = denorm_channel(gx, gfs_mesh, iz500) / 9.80665

print(f"2m temperature : {t2m.min()-273.15:6.1f} .. {t2m.max()-273.15:6.1f} degC (mean {t2m.mean()-273.15:.2f})")
print(f"500 hPa height : {z500.min():6.0f} .. {z500.max():6.0f} m")
print(f"normed pr block: mean {gx[..., :gfs_mesh.n_pr].mean():+.3f}  std {gx[..., :gfs_mesh.n_pr].std():.3f}")

## Load WeatherMesh-3

In [ ]:
from model import get_WeatherMesh3

model = get_WeatherMesh3("model/WeatherMesh3.pt").to(device).eval()
print("parameters:", f"{sum(p.numel() for p in model.parameters())/1e6:.1f} M")

## Run the forecast
`todo=[6]` expands to `E,P6,D` (one 6-hour step). Encoders run in parallel and blend `0.1*gfs + 0.9*hres`.

In [ ]:
x = [gx[None].to(device), hx[None].to(device), t0s.to(device)]
with torch.no_grad():
    out = model(x, [6])

pred = out[6][0]
print("prediction:", tuple(pred.shape), "| latent_l2:", float(out["latent_l2"]))
print("valid time:", get_date(TS + 6 * 3600).strftime("%Y-%m-%d %H:%MZ"))

## Decode the ERA5 output
Absolute predicted state (not a delta), laid out as `era_mesh.full_varlist` including the 13 extra output variables.

In [ ]:
mean = era_mesh.normalization_matrix_mean
std = era_mesh.normalization_matrix_std
real = pred[0].float().cpu().numpy() * std + mean


def field(name):
    return real[..., era_mesh.full_varlist.index(name)]


print("surface output vars:", era_mesh.sfc_vars)
print(f"forecast 2m temp : {field('167_2t').min()-273.15:.1f} .. {field('167_2t').max()-273.15:.1f} degC")
print(f"forecast mslp    : {field('151_msl').min()/100:.0f} .. {field('151_msl').max()/100:.0f} hPa")

## Visualize the forecast
Coastline-referenced fields (land mask, no cartopy) with robust percentile color limits: 2m temperature, MSLP with isobars, 500 hPa geopotential height, 250 hPa jet, large-scale precip (log-detransformed to mm), and the 6h increment. Precip is stored in log space (`142_lsp` mean ~ -13), so it is exponentiated back before plotting.

In [ ]:
EXT = [0, 360, -90, 90]
LM = np.load("constants/additional_variables/land_mask.npy")
LMX = np.linspace(0, 360, LM.shape[1])
LMY = np.linspace(90, -90, LM.shape[0])


def clim(a, lo=2, hi=98, sym=False):
    vlo, vhi = np.nanpercentile(np.asarray(a, np.float32), [lo, hi])
    if sym:
        m = float(max(abs(vlo), abs(vhi)))
        return -m, m
    return float(vlo), float(vhi)


def panel(ax, f, title, cmap, vmin=None, vmax=None, isolines=None):
    f = np.asarray(f, np.float32)
    if vmin is None:
        vmin, vmax = clim(f)
    im = ax.imshow(f, extent=EXT, origin="upper", aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    ax.contour(LMX, LMY, LM, levels=[0.5], colors="k", linewidths=0.4)
    if isolines is not None:
        xs = np.linspace(0, 360, f.shape[1])
        ys = np.linspace(90, -90, f.shape[0])
        cs = ax.contour(xs, ys, f, levels=isolines, colors="k", linewidths=0.5, alpha=0.6)
        ax.clabel(cs, inline=True, fontsize=6, fmt="%d")
    ax.set_xlim(0, 360)
    ax.set_ylim(-90, 90)
    ax.set_title(title)
    plt.colorbar(im, ax=ax, shrink=0.8)


t2 = field("167_2t") - 273.15
mslp = field("151_msl") / 100
z500 = field("129_z_500") / 9.80665
jet = np.sqrt(field("131_u_250") ** 2 + field("132_v_250") ** 2)
precip_mm = np.exp(field("142_lsp")) * 1000.0
incr = t2 - (denorm_channel(gx, gfs_mesh, i2t)[:720] - 273.15)

fig, ax = plt.subplots(3, 2, figsize=(16, 13))
panel(ax[0, 0], t2, "2m temperature (degC)", "RdBu_r")
panel(ax[0, 1], mslp, "MSLP (hPa) + isobars", "viridis", isolines=np.arange(960, 1044, 8))
panel(ax[1, 0], z500, "500 hPa geopotential height (m)", "turbo", isolines=np.arange(4800, 6001, 60))
panel(ax[1, 1], jet, "250 hPa wind speed (m/s, jet)", "plasma", 0, clim(jet)[1])
panel(ax[2, 0], precip_mm, "large-scale precip (mm)", "Blues", 0, clim(precip_mm, hi=99)[1])
panel(ax[2, 1], incr, "6h increment: ERA5 fcst - GFS t0 (degC)", "coolwarm", *clim(incr, sym=True))
fig.tight_layout()
plt.show()

## Notes & caveats
- **Reconstructed input.** The repo withholds its data-processing; the mesh build, input reconstruction and de-norm checks above validate the shapes/ordering independently.
- **No verification target.** This sample has only the `t0` analysis inputs — no ERA5 truth at `t0+6h` — so only plausibility and the forecast-vs-analysis tendency, not RMSE.
- **Interpolation is done in normalized space**, as in training.
- **The `extra/` files are byte-identical across `neogfs` and `neohres`** in this sample.
- **Memory.** Full 720x1440 at `latent_size=1024` is large; use `.half()` or `checkpoint_type='matepoint'` if you hit OOM.